In [1]:
from google.colab import files
uploaded=files.upload()

Saving samsum-test.csv to samsum-test.csv
Saving samsum-train.csv to samsum-train.csv
Saving samsum-validation.csv to samsum-validation.csv


In [3]:
!pip install transformers

In [5]:
!pip install "transformers[torch]"

In [2]:
import pandas as pd
from transformers import T5Tokenizer,Trainer,TrainingArguments,T5ForConditionalGeneration

In [4]:
train_data=pd.read_csv("/content/samsum-train.csv")
val_data=pd.read_csv("/content/samsum-validation.csv")

In [5]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [6]:
train_data['dialogue'][0]

"Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)"

In [7]:
train_data.sample(10)

,id,dialogue,summary
679,13715794,"Olivia: Hi Michael, is everything okay with th...",Olivia and Diego can't log into the system. Th...
3465,13814171,Alice: so how is London kiddo?\r\nJames: hey m...,James is in London. It's cold. He saw Buckingh...
722,13681690,"Ula: Hey, you can kiss my ass!\r\nMolly: What?...",Ula feels humiliated by the behaviour of Molly...
9574,13681815,"Mary: Where are you??\r\nBradley: Home, why?\r...",Bradley will meet with Mary in 15 minutes. The...
3433,13809883,Charlotte: you know it was my birtday?\r\nPete...,Charlotte had a birthday. Peter don't have any...
6107,13828440,"Joanne: hi, wanna talk?\r\nBradley: hi, sorry,...","Joanne has something to tell Bradley, but he's..."
11,13729168,Mark: I just shipped the goods\r\nMark: Tomorr...,Mark just shipped the goods and he will send G...
685,13810066,Diana: hello\r\nMax: hi\r\nDiana: i have some ...,Diana wants Max to meet her parents. He thinks...
11778,13828982,Eliana: Have you called me while sitting on th...,Declan called Eliana when he was in the toilet.
10382,13828914,Adeline: Hi! :) Are you busy rn?\r\nNaomi: hi!...,"Adeline's boyfriend, Matt, doesn't want to mov..."


In [8]:
train_data.shape

(14732, 3)

In [9]:
val_data.shape

(818, 3)

In [10]:
#random sampling
train_data=train_data.sample(n=4000,random_state=42).reset_index(drop=True)
val_data=val_data.sample(n=500,random_state=42).reset_index(drop=True)

In [11]:
train_data.shape

(4000, 3)



```
DATA PREPROCESSING
```



In [12]:
import re

def clean_data(text):
  text=re.sub(r"\r\n"," ",text)
  text=re.sub(r"\s+"," ",text)
  text=re.sub(r"<.*?>"," ",text)
  text=text.strip().lower()
  return text

In [13]:
train_data.columns
train_data['dialogue']=train_data['dialogue'].apply(clean_data)
train_data['summary']=train_data['summary'].apply(clean_data)

val_data['dialogue']=val_data['dialogue'].apply(clean_data)
val_data['summary']=val_data['summary'].apply(clean_data)

Tokenize

In [14]:
tokenizer=T5Tokenizer.from_pretrained("t5-small")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [26]:
def tokenize(data):
  inputs=tokenizer(data['dialogue'],padding="max_length",max_length=512,truncation=True)
  targets=tokenizer(data['summary'],padding="max_length",max_length=150,truncation=True)

  inputs["labels"]=targets["input_ids"]
  return inputs

In [27]:
train_dataset=train_data.apply(tokenize,axis=1).tolist()
val_dataset=val_data.apply(tokenize,axis=1).tolist()

In [28]:
train_dataset[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [29]:
len(train_dataset[0]['input_ids'])

512

In [30]:
type(train_dataset)
type(val_dataset)

list

Model

In [31]:
model=T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [32]:
import torch

if(torch.cuda.is_available()):
  device=torch.device("cuda")
elif(torch.backends.mps.is_available()):
  device=torch.device("mps")
else:
  device=torch.device("cpu")

print("device is ",device)
model.to(device)

device is  cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [33]:
train_args=TrainingArguments(
    output_dir="./results",

    num_train_epochs=6,
    weight_decay=0.01,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    eval_strategy="epoch",
    save_strategy="epoch",

    warmup_steps=500
)

In [34]:
trainer=Trainer(
    model=model,
    args=train_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [35]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,3.577381,0.380097
2,0.397485,0.359731
3,0.374499,0.354865
4,0.362143,0.350373
5,0.355740,0.349276
6,0.350639,0.349105


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.9029809265136719, metrics={'train_runtime': 1184.8502, 'train_samples_per_second': 20.256, 'train_steps_per_second': 2.532, 'total_flos': 3248203235328000.0, 'train_loss': 0.9029809265136719, 'epoch': 6.0})

In [36]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model/tokenizer_config.json',
 './saved_summary_model/tokenizer.json')

In [48]:
model=T5ForConditionalGeneration.from_pretrained("./saved_summary_model").to(device)
tokenizer=T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Testing

In [53]:
def summarize_dialogue(dialogue):
  dialogue=clean_data(dialogue)

  inputs=tokenizer(
      dialogue,
      padding="max_length",
      max_length=32,
      truncation=True,
      return_tensors="pt"
  ).to(device)

  targets=model.generate(
      input_ids=inputs['input_ids'],
      attention_mask=inputs['attention_mask'],
      max_length=150,
      num_beams=4,
      early_stopping=True
  )

  summary=tokenizer.decode(targets[0],skip_special_tokens=True)
  return summary

In [54]:
test_dialogue="""
Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.

Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. However, this growth has also raised questions about job displacement and ethical concerns.

Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks such as language understanding, image recognition, and even code generation.

Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency is becoming a key area of research.

Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies. The goal is to balance innovation with accountability.

Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, operate as “black boxes,” making it difficult to understand how decisions are made.

Reporter: Experts also highlight the importance of responsible AI development, including data privacy, security, and long-term societal impact.

Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be crucial to ensure that AI systems are developed and used in a safe and beneficial way.
"""

summary=summarize_dialogue(test_dialogue)
print("Final Summary:\n",summary)

Final Summary:
 new reports suggest that artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education.


In [56]:
!zip -r NLP_MProject.zip \
results\
sample_data\
saved_summary_model\
samsum-test.csv\
samsum-train.csv\
samsum-validation.csv

  adding: results/ (stored 0%)
  adding: results/checkpoint-1000/ (stored 0%)
  adding: results/checkpoint-1000/model.safetensors (deflated 10%)
  adding: results/checkpoint-1000/scheduler.pt (deflated 61%)
  adding: results/checkpoint-1000/training_args.bin (deflated 53%)
  adding: results/checkpoint-1000/config.json (deflated 63%)
  adding: results/checkpoint-1000/rng_state.pth (deflated 26%)
  adding: results/checkpoint-1000/generation_config.json (deflated 29%)
  adding: results/checkpoint-1000/optimizer.pt (deflated 7%)
  adding: results/checkpoint-1000/trainer_state.json (deflated 63%)
  adding: results/checkpoint-2000/ (stored 0%)
  adding: results/checkpoint-2000/model.safetensors (deflated 10%)
  adding: results/checkpoint-2000/scheduler.pt (deflated 61%)
  adding: results/checkpoint-2000/training_args.bin (deflated 53%)
  adding: results/checkpoint-2000/config.json (deflated 63%)
  adding: results/checkpoint-2000/rng_state.pth (deflated 26%)
  adding: results/checkpoint-2000/

In [60]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [61]:
!cp NLP_MProject.zip /content/drive/MyDrive